In [185]:
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.features import GeoJsonTooltip
import matplotlib.patches as mpatches


In [228]:
class RegressionLogistique:
    def __init__(self, taux_apprentissage, n):
        self.taux_apprentissage = taux_apprentissage
        self.n = n
        self.poids = None
        self.biais = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def descente_gradient(self, X, Y):
        n_observation, n_var_explicative = X.shape

        # On initialise tout à 0 pour la descente de gradient
        self.poids = np.zeros(n_var_explicative)
        self.biais = 0

        # Descente de gradient
        for _ in range(self.n):
            modele = np.dot(X, self.poids) + self.biais
            Y_predit = self.sigmoid(modele)  # Pour avoir les probabilités

            # On calcule les gradients des poids et du biais 
            # Les formules viennent de la dérivée de la fonction de coût (log-loss pour la régression logistique)
            # par rapport aux poids et au biais
            gradient_poids = (1 / n_observation) * np.dot(X.T, (Y_predit - Y))  #X.T désigne la transposée de X
            gradient_biais = (1 / n_observation) * np.sum(Y_predit - Y)

            # On met à jour les poids et le biais
            self.poids -= self.taux_apprentissage * gradient_poids
            self.biais -= self.taux_apprentissage * gradient_biais

    def proba_estimee(self, X):
        modele = np.dot(X, self.poids) + self.biais
        return self.sigmoid(modele)


In [204]:
#Chargement des données
Crimes_communes = pd.read_csv("/Users/adriensorin/Desktop/Documents ENSAE/2A/Python/Données/CSP/crimesdelitscommunes.csv")
Population_communes = pd.read_csv("/Users/adriensorin/Desktop/Documents ENSAE/2A/Python/Données/Population/popcommunes.csv")

#On ne garde que la variable qui donne le nombre d'habitants de la commune en 2022
Population_communes_2022 = Population_communes[['dep', 'nomdep', 'codecommune', 'nomcommune', 'pop2022']]

#Filtrer pour le Finistère
Population_communes_2022 = Population_communes_2022[Population_communes_2022['nomdep'] == 'FINISTERE']

#Filtrer les crimes pour le Finistère et ne garder que les colonnes de 2020
Crimes_communes = Crimes_communes[Crimes_communes['nomdep'] == 'FINISTERE']
Crimes_communes_suffix = Crimes_communes.filter(regex="2020$")  # colonnes 2020
Crimes_communes_2022 = Crimes_communes[["dep", "nomdep", "codecommune", "nomcommune"]].join(Crimes_communes_suffix)


/Users/adriensorin/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0,2) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [205]:
csp_communes = pd.read_csv("/Users/adriensorin/Desktop/Documents ENSAE/2A/Python/Données/CSP/cspcommunes.csv")

#On met tous les codes communes à 5 chiffres, et on sélectionne uniquement celles du Finistère (département 29)
csp_communes['codecommune'] = csp_communes['codecommune'].astype(str).str.zfill(5)
csp_communes_finistere = csp_communes[csp_communes['nomdep'] == 'FINISTERE']
csp_communes_suffix = csp_communes_finistere.filter(regex="2022$")  #on étudie pour l'instant en 2022, on ne garde que les variables de 2022
csp_communes_2022 = csp_communes_finistere[["dep", "nomdep", "codecommune", "nomcommune"]].join(csp_communes_suffix)


In [206]:
PIB_communes = pd.read_csv("/Users/adriensorin/Desktop/Documents ENSAE/2A/Python/Données/Revenus/pibcommunes.csv")


#On filtre en ne gardant que le Finistère et les colonnes qui nous intéressent 
PIB_communes = PIB_communes[PIB_communes['nomdep'] == 'FINISTERE']
PIB_communes_suffix = PIB_communes.filter(regex="2022$")  #on étudie pour l'instant en 2022, on ne garde que les variables de 2022
PIB_communes_2022 = PIB_communes[["dep", "nomdep", "codecommune", "nomcommune"]].join(PIB_communes_suffix)


In [207]:
# On enlève les colonnes en double avant de merger pour avoir la base totale

Population_communes_2022 = Population_communes_2022.drop(columns=["dep", "nomcommune", "nomdep"])
csp_communes_2022 = csp_communes_2022.drop(columns=["dep", "nomcommune", "nomdep"])
PIB_communes_2022 = PIB_communes_2022.drop(columns=["dep", "nomcommune", "nomdep", "pop2022"])


In [208]:
base_totale = pd.merge(Crimes_communes_2022, Population_communes_2022, left_on = 'codecommune', right_on = 'codecommune',how='inner')
base_totale = pd.merge(base_totale, csp_communes_2022, left_on = 'codecommune', right_on = 'codecommune',how='inner')
base_totale = pd.merge(base_totale, PIB_communes_2022, left_on = 'codecommune', right_on = 'codecommune',how='inner')

In [209]:
# On crée et ajoute la variable de l'évolution du vote du RN entre 2017 et 2022

data2017 = pd.read_csv("donnees/élections/leg2017comm.csv", delimiter=",", low_memory=False)
data2017 = data2017[data2017["dep"] == "29"]
data2017["codecommune"] = data2017["codecommune"].astype(int)

data2022 = pd.read_csv("donnees/élections/leg2022comm.csv", delimiter=",", low_memory=False)
data2022 = data2022[data2022["dep"] == "29"]
data2022["codecommune"] = data2022["codecommune"].astype(int)

In [210]:
# Pour rendre les deux bases de la même taille
data2017 = data2017[data2017["codecommune"] != 29129]
data2017 = data2017[data2017["codecommune"] != 29119]

data2022["VarRN"] = data2022["pvoixRN"].values - data2017["pvoixFN"].values

data2022 = data2022[["codecommune", "VarRN"]]

base_totale["codecommune"] = base_totale["codecommune"].astype(int)




In [211]:
base_totale = pd.merge(base_totale, data2022, left_on = 'codecommune', right_on = 'codecommune',how='inner')

In [212]:
# On crée la variable cible binaire qui vaut 1 si la variation du vote pour le RN est supérieure à la moyenne, et 0 sinon

base_totale["VarRN"] = (base_totale["VarRN"] > base_totale["VarRN"].mean()).astype(int)
Y = base_totale["VarRN"]

In [213]:
# On choisit les variables explicatives :
# Nombre de crimes et délits, pourcentage d'ouvriers, population totale et pib total de chaque commune.

var_explicatives = [
    "ncrimesdelits2020",
    "pouvr2022",
    "pop2020",
    "pibtot2022"
]

X = base_totale[var_explicatives]

In [214]:
# On normalise les données

X_norm = (X - X.mean()) / X.std()

In [215]:
# Pour avoir un tableau numpy

X_norm = X_norm.values


In [237]:
# On applique désormais la régression logistique. On prend un taux d'apprentissage de 0.01, qui est un compromis
# intéressant entre vitesse de conergence et précision de la prédiction. On prend n = 277 car il y a 277 communes
# du Finistère dans notre base de données. 

regression = RegressionLogistique(taux_apprentissage = 0.01, n = 277)

regression.descente_gradient(X_norm, Y)

In [244]:
tableau_coeff = pd.DataFrame({
    "variable": var_explicatives,
    "coefficient": regression.poids,
    "odds_ratio": np.exp(regression.poids)
})

print(tableau_coeff)


            variable  coefficient  odds_ratio
0  ncrimesdelits2020    -0.055890    0.945644
1          pouvr2022     0.084855    1.088559
2            pop2020    -0.091180    0.912853
3         pibtot2022    -0.125151    0.882363


In [245]:
# On cherche ici à déterminer l'impact d'une hausse des variables explicatives sur la probabilité de valoir 1 pour Y
# c'est-à-dire sur la probabilité d'avoir une forte augmentation pour le vote d'extrême-droite.

# Pour la probabilité initiale, on prend la moyenne des probabilités (ie le nombre de communes pour lequelles Y = 1 sur le nombre total de communes)
P0 = 0.469  


# Fonction qui donne la probabilité nouvelle d'avoir Y = 1 après une augmentation de une unité de la variable explicative 
def proba_apres_variation(P0, odds_ratio):
    odds0 = P0 / (1 - P0)
    odds1 = odds0 * odds_ratio
    P1 = odds1 / (1 + odds1)
    return P1


# On initialise une nouvelle colonne 
tableau_coeff["proba_apres_variation"] = 0.0

# On applique la fonction proba_apres_variation à toutes les lignes
for i in range(len(tableau_coeff)):
    odds_ratio = tableau_coeff.loc[i, "odds_ratio"]   
    tableau_coeff.loc[i, "proba_apres_variation"] = proba_apres_variation(P0, odds_ratio)  


# On crée enfin la variable qui donne la variation de la probabilité 
tableau_coeff["variation_proba"] = tableau_coeff["proba_apres_variation"] - P0

print(tableau_coeff)


            variable  coefficient  odds_ratio  proba_apres_variation  \
0  ncrimesdelits2020    -0.055890    0.945644               0.455109   
1          pouvr2022     0.084855    1.088559               0.490175   
2            pop2020    -0.091180    0.912853               0.446372   
3         pibtot2022    -0.125151    0.882363               0.437993   

   variation_proba  
0        -0.013891  
1         0.021175  
2        -0.022628  
3        -0.031007  


In [ ]:
""" 

Analyse des résultats obtenus :

Dans cette partie, nous avons effectué une régression logistique afin de déterminer quels sont les liens entre certaines
variables et la montée du vote d'extrême droite. 

Nous avons sélectionné 4 varibales explicatives : le nombre de crimes et de délits, le pourcentage d'ouvriers,
la population totale, ainsi que le pib total de chaque commune.
La variable cible Y (celle qu'on veut expliquer) correspond à la variation de pourcentage du vote pour le Rassemblement
National entre 2017 et 2022. Y = 1 signifie que la variation est importante, Y = 0 signifie que la variation est modérée ou faible.

Le modèle donne alors les coefficients de chaque variable. Un coefficient positif signifie que, si la variable augmente,
alors la probabilité que la commune vote plus pour le RN augmente. C'est le cas pour le coefficient associé à la variable
correspondant au pourcentage du nombre d'ouvriers. Ceci est cohérent avec la réalité et était attendu : plus la proportion d'ouvriers
dans la commune augmente, plus la commune a tendance à voter davantage pour le RN. Ce sont en effet les classes dites populaires qui
votent majoritairement pour le parti d'extrême droite. Par ailleurs, le coefficient est assez faible en valeur absolue, ce qui signifie
que l'impact n'est pas très fort.

Pour les autres variables, le coefficient est négatif : une augmentation de la variable implique une diminution de la probabilité
que Y = 1, c'est-à-dire une diminution de la probabilité que la commune ait une forte augmentation de vote pour le RN. 
Pour le nombre de crimes, ceci peut s'expliquer car les communes où les crimes sont les plus élevés sont des grandes communes (plus forte
population et plus haut pib), et ces dernières sont moins enclines à voter plus pour l'extrême droite. 
Pour la population, ceci peut s'expliquer car une commune avec une plus grande population est une plus grosse commune, donc
probablement avec un pib plus important aussi, et potentiellement moins d'ouvriers. Le vote d'extrême droite est alors moins 
important.
Les coefficients liés au nombre de crimes et à la population sont, en valeur absolue, du même ordre de grandeur que la proportion
d'ouvriers. 
Enfin, le coefficient lié au PIB est négatif : plus une commune a un PIB élevé, moins elle a tendance à avoir une varition
forte de vote pour le RN. En effet, on constate dans les études sur le vote que les villes les plus riches votent moins à l'extrême droite
que les villes les plus pauvres. Ce coefficient est le plus fort en valeur absolue : l'effet du pib est plus important que celui des autres variables.


Enfin, nous avons obtenu l'impact en point de probabilité de l'augmentation d'une unité de chaque variable.
Par exemple, pour la proportion d'ouvriers, l'impact est de 0.021 : si la proportion d'ouvriers augmente de 1%,
alors la probabilité que la commune ait une forte variation du vote pour le RN augmente de 2.1%. 
Cette probabilité diminue pour les autres variables, ce qui est normal en raison des signes déjà analysés des coefficients de la régression.

"""


